Fine-tuning модели на задаче обнаружения аномалий в логах

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from datasets import Dataset
import matplotlib.pyplot as plt
import random
from collections import Counter

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
from peft import PeftModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# Конфигурация
RANDOM_SEED = 42
MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"
DATASET_PATH = "./hdfs_train.jsonl"
TEST_PATH = "./hdfs_test.jsonl"
OUTPUT_DIR = "./qwen_hdfs_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")

Загрузка HDFS датасета из jsonl

In [ ]:
def load_hdfs_data(file_path, max_samples=None):
    data = []
    with open(file_path, 'r') as f:
        for i, line in enumerate(f):
            if max_samples and i >= max_samples:
                break
            item = json.loads(line)
            data.append({
                'text': item['input'],
                'label': 1 if item['output'] == 'Anomaly' else 0,
                'label_text': item['output']
            })
    return data

Создание сбалансированной выборки

In [ ]:
def create_balanced_sample(data, target_size, target_ratio=0.5):

    normal_data = [item for item in data if item['label'] == 0]
    anomaly_data = [item for item in data if item['label'] == 1]

    n_anomaly = int(target_size * target_ratio)
    n_normal = target_size - n_anomaly

    if len(anomaly_data) < n_anomaly:
        print(f" Недостаточно аномалий! Берем все {len(anomaly_data)}")
        n_anomaly = len(anomaly_data)
        n_normal = target_size - n_anomaly

    if len(normal_data) < n_normal:
        print(f" Недостаточно нормальных! Берем все {len(normal_data)}")
        n_normal = len(normal_data)
        n_anomaly = target_size - n_normal

    sampled_normal = random.sample(normal_data, n_normal) if n_normal > 0 else []
    sampled_anomaly = random.sample(anomaly_data, n_anomaly) if n_anomaly > 0 else []

    balanced_data = sampled_normal + sampled_anomaly
    random.shuffle(balanced_data)

    print(f"  Создано: нормальных={len(sampled_normal)}, аномалий={len(sampled_anomaly)}")
    print(f"  Соотношение: {len(sampled_anomaly)/len(balanced_data)*100:.1f}% аномалий")

    return balanced_data

In [ ]:
all_train_data = load_hdfs_data(DATASET_PATH)
all_test_data = load_hdfs_data(TEST_PATH)

train_labels = [d['label'] for d in all_train_data]
test_labels = [d['label'] for d in all_test_data]


train_data = create_balanced_sample(
    all_train_data,
    target_size=100000,
    target_ratio=0.5  # 50% аномалий, 50% нормальных
)

test_data = create_balanced_sample(
    all_test_data,
    target_size=5000,
    target_ratio=0.5  # 50% аномалий, 50% нормальных
)

Загрузка базовой модели через Unsloth

In [ ]:
def load_base_model():

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=2048,
        dtype=None,
        load_in_4bit=True,
        token=None,
    )

    return model, tokenizer

base_model, tokenizer = load_base_model()

Подготовка данных для тестирования до FT

In [ ]:
def prepare_test_prompts(test_data):
    prompts = []
    for item in test_data:
        messages = [
            {"role": "system", "content": "Ты - эксперт по анализу логов HDFS. Определи, является ли лог аномальным. Отвечай только 'OK' или 'Anomaly'."},
            {"role": "user", "content": item['text']}
        ]

        #использование apply_chat_template как в примере Unsloth
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        prompts.append({
            'prompt': prompt,
            'true_label': item['label'],
            'true_label_text': item['label_text'],
            'text': item['text']
        })
    return prompts

Функция для тестирования

In [ ]:
def test_model(model, tokenizer, test_prompts, max_new_tokens=10, sample_size=500):

    model.eval()
    predictions = []
    true_labels = []

    test_subset = test_prompts[:sample_size]

    for i, item in enumerate(test_subset):
        messages = [
            {"role": "system", "content": "Ты - эксперт по анализу логов HDFS. Определи, является ли лог аномальным. Отвечай только 'OK' или 'Anomaly'."},
            {"role": "user", "content": item['text']}
        ]

        #подготовка данных
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        #генерация ответа
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        #декодировщик
        response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        response = response.strip().lower()

        if 'anomaly' in response:
            pred = 1
        elif 'ok' in response:
            pred = 0
        else:
            if response.startswith(('anomaly', 'error', 'fail')):
                pred = 1
            else:
                pred = 0

        predictions.append(pred)
        true_labels.append(item['true_label'])

    return predictions, true_labels

test_prompts = prepare_test_prompts(test_data)

Тестирование базовой модели

In [ ]:
base_pred, base_true = test_model(base_model, tokenizer, test_prompts, sample_size=500)

base_accuracy = accuracy_score(base_true, base_pred)
base_precision = precision_score(base_true, base_pred, average='binary', zero_division=0)
base_recall = recall_score(base_true, base_pred, average='binary', zero_division=0)
base_f1 = f1_score(base_true, base_pred, average='binary', zero_division=0)

print("МЕТРИКИ БАЗОВОЙ МОДЕЛИ:")
print(f"  Accuracy:  {base_accuracy:.4f}")
print(f"  Precision: {base_precision:.4f}")
print(f"  Recall:    {base_recall:.4f}")
print(f"  F1-score:  {base_f1:.4f}")

base_results = {
    'model': MODEL_NAME,
    'stage': 'base',
    'accuracy': base_accuracy,
    'precision': base_precision,
    'recall': base_recall,
    'f1': base_f1,
}

with open(os.path.join(OUTPUT_DIR, 'base_model_results.json'), 'w') as f:
    json.dump(base_results, f, indent=2)

Функция для подготовки данных для обучения

In [ ]:
def prepare_train_dataset(train_data, tokenizer):

    formatted_data = []
    for i, item in enumerate(train_data):
        messages = [
            {"role": "system", "content": "Ты - эксперт по анализу логов HDFS. Определи, является ли лог аномальным. Отвечай только 'OK' или 'Anomaly'."},
            {"role": "user", "content": item['text']},
            {"role": "assistant", "content": item['label_text']}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        formatted_data.append({"text": text})

        if (i + 1) % 10000 == 0:
            print(f"  Подготовлено {i + 1}/{len(train_data)} примеров...")

    return Dataset.from_list(formatted_data)

Fine-tuning модели

In [ ]:
def fine_tune_model(base_model, tokenizer, train_data, output_dir):

    #подготовка модели для LoRA
    model = FastLanguageModel.get_peft_model(
        base_model,
        r=16,  # ранг LoRA
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"],
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=RANDOM_SEED,
        max_seq_length=2048,
    )

    #подготавка датасета
    train_dataset = prepare_train_dataset(train_data, tokenizer)

    #гиперпараметры обучения
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=50,
        save_steps=500,
        save_total_limit=2,
        remove_unused_columns=False,
        report_to="none",
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        args=training_args,
        max_seq_length=2048,
        dataset_text_field="text",
    )

    trainer.train()

    #сохранение
    model.save_pretrained(os.path.join(output_dir, "final_model"))
    tokenizer.save_pretrained(os.path.join(output_dir, "final_model"))
    return model

Проведение FT

In [ ]:
ft_model = fine_tune_model(
    base_model,
    tokenizer,
    train_data[:10000],
    os.path.join(OUTPUT_DIR, "ft_checkpoints")
)

Тестирование FT-модели

In [ ]:
ft_pred, ft_true = test_model(ft_model, tokenizer, test_prompts, sample_size=500)

ft_accuracy = accuracy_score(ft_true, ft_pred)
ft_precision = precision_score(ft_true, ft_pred, average='binary', zero_division=0)
ft_recall = recall_score(ft_true, ft_pred, average='binary', zero_division=0)
ft_f1 = f1_score(ft_true, ft_pred, average='binary', zero_division=0)

print("\n МЕТРИКИ FINE-TUNED МОДЕЛИ:")
print(f"  Accuracy:  {ft_accuracy:.4f}")
print(f"  Precision: {ft_precision:.4f}")
print(f"  Recall:    {ft_recall:.4f}")
print(f"  F1-score:  {ft_f1:.4f}")

ft_results = {
    'model': MODEL_NAME,
    'stage': 'fine_tuned',
    'accuracy': ft_accuracy,
    'precision': ft_precision,
    'recall': ft_recall,
    'f1': ft_f1,
}

with open(os.path.join(OUTPUT_DIR, 'ft_model_results.json'), 'w') as f:
    json.dump(ft_results, f, indent=2)

Сравнение результатов

In [ ]:
comparison = pd.DataFrame({
    'Метрика': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Базовая модель': [base_accuracy, base_precision, base_recall, base_f1],
    'Fine-tuned модель': [ft_accuracy, ft_precision, ft_recall, ft_f1],
    'Улучшение': [
        ft_accuracy - base_accuracy,
        ft_precision - base_precision,
        ft_recall - base_recall,
        ft_f1 - base_f1
    ]
})

print("\n" + comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score']
x = np.arange(len(metrics))
width = 0.35

axes[0].bar(x - width/2, [base_accuracy, base_precision, base_recall, base_f1],
            width, label='Базовая', color='skyblue')
axes[0].bar(x + width/2, [ft_accuracy, ft_precision, ft_recall, ft_f1],
            width, label='Fine-tuned', color='lightcoral')

axes[0].set_xlabel('Метрики')
axes[0].set_ylabel('Значение')
axes[0].set_title('Сравнение метрик')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics)
axes[0].legend()
axes[0].set_ylim(0, 1.0)

#добавление значений на столбцы
for i, v in enumerate([base_accuracy, base_precision, base_recall, base_f1]):
    axes[0].text(i - width/2, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
for i, v in enumerate([ft_accuracy, ft_precision, ft_recall, ft_f1]):
    axes[0].text(i + width/2, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)

#график улучшений
improvements = comparison['Улучшение'].values
colors = ['green' if x > 0 else 'red' for x in improvements]

axes[1].bar(metrics, improvements, color=colors)
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].set_xlabel('Метрики')
axes[1].set_ylabel('Улучшение')
axes[1].set_title('Улучшение после fine-tuning')

#добавление значений
for i, v in enumerate(improvements):
    axes[1].text(i, v + (0.01 if v > 0 else -0.03), f'{v:+.3f}',
                ha='center', va='bottom' if v > 0 else 'top', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

comparison.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison.csv'), index=False)

avg_improvement = np.mean(improvements)
if avg_improvement > 0.05:
    print(f" Fine-tuning показал значительное улучшение (+{avg_improvement:.3f} в среднем)")
elif avg_improvement > 0:
    print(f" Fine-tuning показал небольшое улучшение (+{avg_improvement:.3f} в среднем)")
else:
    print(f" Fine-tuning не дал улучшения ({avg_improvement:.3f} в среднем)")

print(f"\nЛучше всего улучшилась метрика: {metrics[np.argmax(improvements)]} (+{max(improvements):.3f})")

Сохранение LoRA адаптера (для model merging)

In [ ]:
lora_path = os.path.join(OUTPUT_DIR, "hdfs_lora_adapter")
ft_model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)